# 🎯 Career Prediction — Multimodal Ensemble

| # | Model | File | Architecture | Data |
|---|-------|------|--------------|------|
| 1 | **Personality + Skills** | `calibrated_lgbm_model.pkl` | Calibrated LightGBM | Real survey data |
| 2 | **Skills Profile** | `career_model.pkl` | Calibrated XGBoost | Generated (reference — replace when real data arrives) |

**How input works:**
- The user sees every sub-skill that exists inside `feature_schema` and rates it **1–10**
- Ratings are converted to proficiency levels (0–4) and stored in the DB
- For **Model 2**: any skill rated ≥ 3 (Beginner+) is treated as *present* — the exact sub-skill string is matched against `feature_schema` groups, exactly replicating `score_row()` from training
- For **Model 1**: the four numeric features (Computer Architecture, Programming Skills, Project Management, Communication skills) come from the matching sub-skill ratings; Big-Five traits are entered as raw 0.0–1.0 floats

| Rating | Proficiency | Stored as |
|--------|-------------|----------|
| 1 – 2  | None        | none |
| 3 – 5  | Beginner    | beginner |
| 6 – 7  | Intermediate| intermediate |
| 8 – 9  | Advanced    | advanced |
| 10     | Mastered    | mastered |

## 0. Install / Import Dependencies

In [4]:
!pip install lightgbm xgboost scikit-learn pandas numpy --quiet

In [5]:
import pickle
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

print('Dependencies loaded ✓')

Dependencies loaded ✓


## 1. Load Both Models & Their Artefacts

In [6]:
# ─── Model 1 (Calibrated LightGBM — CareerMapping1.csv) ──────────────────────
MODEL1_PATH   = '/content/calibrated_lgbm_model.pkl'
SCALER_PATH   = '/content/scaler.pkl'
LE1_PATH      = '/content/label_encoder.pkl'         # rename if saved separately e.g. le_lgbm.pkl

# ─── Model 2 (Calibrated XGBoost — Career Dataset.csv) ───────────────────────
MODEL2_PATH          = '/content/career_model.pkl'
LE2_PATH             = '/content/label_encoder1.pkl'  # rename if saved separately e.g. le_xgb.pkl
FEATURE_SCHEMA_PATH  = '/content/feature_schema.pkl'
FEATURE_NAMES_PATH   = '/content/feature_names.pkl'
SKILL_GAP_MAP_PATH   = '/content/skill_gap_map.pkl'
CAREER_PROFILES_PATH = '/content/career_profiles.pkl'

def load_artifact(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Cannot find: {path} — please check your file paths.')
    try:
        return joblib.load(p)
    except Exception:
        with open(p, 'rb') as f:
            return pickle.load(f)

print('Loading Model 1 (Calibrated LightGBM — CareerMapping1)...')
model1  = load_artifact(MODEL1_PATH)
scaler  = load_artifact(SCALER_PATH)
le1     = load_artifact(LE1_PATH)
print(f'  Classes : {list(le1.classes_)}')

print('\nLoading Model 2 (Calibrated XGBoost — Career Dataset)...')
model2          = load_artifact(MODEL2_PATH)
le2             = load_artifact(LE2_PATH)
feature_schema  = load_artifact(FEATURE_SCHEMA_PATH)
feature_names   = load_artifact(FEATURE_NAMES_PATH)
skill_gap_map   = load_artifact(SKILL_GAP_MAP_PATH)
career_profiles = load_artifact(CAREER_PROFILES_PATH)
print(f'  Classes        : {list(le2.classes_)}')
print(f'  Feature groups : {len(feature_names)}')
print(f'  Feature names  : {feature_names}')

print('\nAll artefacts loaded ✓')

Loading Model 1 (Calibrated LightGBM — CareerMapping1)...
  Classes : ['Artificial Intelligence', 'Data Science & Analytics', 'Security', 'Software Development and Engineering', 'User Experience (UX) and User Interface (UI) Design', 'Web and Application Development']

Loading Model 2 (Calibrated XGBoost — Career Dataset)...
  Classes        : ['Artificial Intelligence', 'Data Science', 'Security', 'Software Development and Engineering', 'User Experience (UX) and User Interface (UI) Design', 'Web and Application Development']
  Feature groups : 27
  Feature names  : ['f_ai_ml_core', 'f_nlp_deep_learning', 'f_robotics_iot', 'f_data_analysis', 'f_statistical_analysis', 'f_data_engineering', 'f_data_science', 'f_database', 'f_web_specific', 'f_dev_mobile_android', 'f_dev_mobile_ios', 'f_dev_mobile_general', 'f_dev_game_arvr', 'f_api_testing_qa', 'f_system_design', 'f_cloud', 'f_emerging_tech', 'f_security_cyber', 'f_security_network', 'f_ui_ux_core', 'f_graphic_video', 'f_marketing', 'f_pr

## 2. Collect All Sub-Skills from feature_schema

Instead of hard-coding skill names, we read them **directly from the loaded `feature_schema`**.
This means the survey automatically shows every skill the model was actually trained on — no drift.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Extract every unique sub-skill that appears in feature_schema.
# These are the EXACT strings the model was trained with — we must match them
# precisely at inference time (score_row() does set membership, not fuzzy match).
# ─────────────────────────────────────────────────────────────────────────────

ALL_SCHEMA_SKILLS = sorted(set(
    skill
    for group, sub_skills in feature_schema.items()
    if group != 'f_sparse_row'          # sparse flag has no sub-skills
    for skill in sub_skills
))

print(f'Total unique sub-skills in feature_schema: {len(ALL_SCHEMA_SKILLS)}')
print('(These are the skills shown to the user for rating)\n')
for s in ALL_SCHEMA_SKILLS:
    print(f'  - {s}')

Total unique sub-skills in feature_schema: 78
(These are the skills shown to the user for rating)

  - API Testing and Knowledge
  - AR/VR Development
  - Algorithm Design and Development
  - Android App Development
  - Application Development
  - Artificial Intelligence and Machine Learning
  - Automation
  - Big Data Technologies
  - Blockchain Technology
  - Bot Development
  - Cloud Computing
  - Collaboration
  - Computer Architecture
  - Computer Networking
  - Content Creation
  - Cryptography
  - Cyber Security
  - Data Analysis
  - Data Analysis and Visualization
  - Data Engineering
  - Data Management
  - Data Processing
  - Data Science
  - Data Strategy
  - Data Structure
  - Database Administration
  - Database Development
  - Database Management
  - Deep Learning
  - Digital Art
  - Digital Marketing
  - Education
  - Electronics Engineering
  - Firmware Development
  - Game Development
  - Graphic Design and Editing
  - HTML/CSS
  - Interpersonal Skills
  - IoT and Hard

## 3. Rating System & Skill Lists

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# RATING → PROFICIENCY  (technical skills only, rated 1-10)
# ─────────────────────────────────────────────────────────────────────────────

def rating_to_proficiency(rating) -> dict:

    r = int(rating)
    if r <= 1:   return {'level': 0, 'label': 'None',         'db_value': 'none'}
    elif r <= 5: return {'level': 1, 'label': 'Beginner',     'db_value': 'beginner'}
    elif r <= 7: return {'level': 2, 'label': 'Intermediate', 'db_value': 'intermediate'}
    elif r <= 9: return {'level': 3, 'label': 'Advanced',     'db_value': 'advanced'}
    else:        return {'level': 4, 'label': 'Mastered',     'db_value': 'mastered'}


# ─────────────────────────────────────────────────────────────────────────────
# Big-Five traits — entered as raw 0.0-1.0 floats (IBM Watson / survey score).
# These only feed Model 1. They bypass rating_to_proficiency entirely.
# ─────────────────────────────────────────────────────────────────────────────

BIG_FIVE_TRAITS = [
    'Openness',
    'Conscientousness',
    'Extraversion',
    'Agreeableness',
    'Emotional_Range',
    'Conversation',
    'Openness to Change',
    'Hedonism',
    'Self-enhancement',
    'Self-transcendence',
]

# ─────────────────────────────────────────────────────────────────────────────
# Model 1 also needs these four numeric features derived from skill ratings.
# They are sub-skills that happen to exist in the Career Dataset too,
# but their names must match EXACTLY what Model 1 was trained on.
# ─────────────────────────────────────────────────────────────────────────────

MODEL1_TECH_FEATURES = [
    'Computer Architecture',   # maps from feature_schema sub-skill of same name
    'Programming Skills',      # maps from feature_schema sub-skill of same name
    'Project Management',      # maps from feature_schema sub-skill of same name
    'Communication skills',    # maps from feature_schema sub-skill of same name
]

# Full ordered column list that Model 1 was trained on
MODEL1_COLUMNS = [
    'Computer Architecture',
    'Programming Skills',
    'Project Management',
    'Communication skills',
    'Openness',
    'Conscientousness',
    'Extraversion',
    'Agreeableness',
    'Emotional_Range',
    'Conversation',
    'Openness to Change',
    'Hedonism',
    'Self-enhancement',
    'Self-transcendence',
]

print(f'Sub-skills shown for rating (from feature_schema) : {len(ALL_SCHEMA_SKILLS)}')
print(f'Big-Five traits (0.0-1.0 float input)             : {len(BIG_FIVE_TRAITS)}')
print(f'Model 1 columns                                    : {len(MODEL1_COLUMNS)}')

Sub-skills shown for rating (from feature_schema) : 78
Big-Five traits (0.0-1.0 float input)             : 10
Model 1 columns                                    : 14


## 4. Class Alignment

In [9]:
CANONICAL_CAREERS = [
    'Artificial Intelligence',
    'Data Science & Analytics',
    'Security',
    'Software Development and Engineering',
    'User Experience (UX) and User Interface (UI) Design',
    'Web and Application Development',
]

# Model-1 label → canonical
MODEL1_TO_CANONICAL = {
    'Artificial Intelligence':                             'Artificial Intelligence',
    'Data Science & Analytics':                            'Data Science & Analytics',
    'Security':                                            'Security',
    'Software Development and Engineering':                'Software Development and Engineering',
    'User Experience (UX) and User Interface (UI) Design': 'User Experience (UX) and User Interface (UI) Design',
    'Web and Application Development':                     'Web and Application Development',
}

# Model-2 label → canonical
MODEL2_TO_CANONICAL = {
    'Artificial Intelligence':                             'Artificial Intelligence',
    'Data Science':                                        'Data Science & Analytics',
    'Web and Application Development':                     'Web and Application Development',
    'Security':                                            'Security',
    'Software Development and Engineering':                'Software Development and Engineering',
    'User Experience (UX) and User Interface (UI) Design': 'User Experience (UX) and User Interface (UI) Design',
}

def map_probs_to_canonical(proba_vector, le, class_mapping, canonical_careers):
    canonical     = {c: 0.0 for c in canonical_careers}
    unmapped_mass = 0.0
    for i, cls in enumerate(le.classes_):
        canon = class_mapping.get(cls)
        if canon and canon in canonical:
            canonical[canon] += proba_vector[i]
        else:
            unmapped_mass += proba_vector[i]
    total_mapped = sum(canonical.values())
    if unmapped_mass > 0 and total_mapped > 0:
        for k in canonical:
            canonical[k] += unmapped_mass * (canonical[k] / total_mapped)
    return np.array([canonical[c] for c in canonical_careers])

print('Canonical career classes:')
for i, c in enumerate(CANONICAL_CAREERS):
    print(f'  {i}: {c}')

Canonical career classes:
  0: Artificial Intelligence
  1: Data Science & Analytics
  2: Security
  3: Software Development and Engineering
  4: User Experience (UX) and User Interface (UI) Design
  5: Web and Application Development


## 5. Feature Construction

### Model 2 — `score_row()` replicated exactly

The key fix: the user's ratings are converted to a **set of active skill strings** (those rated ≥ 3 = Beginner+). Then for each feature group in `feature_schema`, we count how many of its sub-skill strings appear in that set — **exactly** what `score_row()` does during training. No hard-coded skill lists needed.

In [10]:
def ratings_to_active_skills(ratings: dict) -> set:
    """
    Convert the user's {skill_name: rating_1_to_10} dict into a set of
    active skill strings — those rated Beginner or above (rating >= 3).

    Big-Five traits are excluded (they are not skill strings in feature_schema).

    This set is then used by score_row_from_active_skills() in exactly the
    same way score_row() used a comma-separated string during training.
    """
    active = set()
    for skill, rating in ratings.items():
        if skill in BIG_FIVE_TRAITS:
            continue
        if rating_to_proficiency(rating)['level'] >= 1:  # Beginner+
            active.add(skill)
    return active


def score_row_from_active_skills(active_skills: set) -> dict:
    """
    Replicates score_row() from training exactly.

    For each feature group in feature_schema:
        score = number of that group's sub-skills present in active_skills

    f_sparse_row = 1 if fewer than 2 skills are active (low-quality input).

    The feature names and their order come from the loaded feature_names
    artefact — guaranteed to match what the model was trained on.
    """
    scores = {}
    for feature, sub_skills in feature_schema.items():
        if feature == 'f_sparse_row':
            continue
        scores[feature] = sum(1 for s in sub_skills if s in active_skills)

    scores['f_sparse_row'] = 1 if len(active_skills) < 2 else 0
    return scores


def build_model2_features(ratings: dict) -> pd.DataFrame:
    """
    Build the feature DataFrame for Model 2 (XGBoost).
    Column order matches feature_names exactly — as loaded from the pkl.
    """
    active_skills = ratings_to_active_skills(ratings)
    scores        = score_row_from_active_skills(active_skills)
    # Enforce the exact column order the model was trained with
    row = {fn: scores.get(fn, 0) for fn in feature_names}
    return pd.DataFrame([row], columns=feature_names)


def build_model1_features(ratings: dict) -> pd.DataFrame:
    """
    Build the 14-column DataFrame for Model 1 (LightGBM).

    Technical features (Computer Architecture, Programming Skills,
    Project Management, Communication skills):
        user 1-10 rating  →  ordinal level 0-4

    Big-Five / values traits:
        raw 0.0-1.0 float  →  passed through unchanged
        (NEVER run through rating_to_proficiency)
    """
    def tech(skill):
        return rating_to_proficiency(ratings.get(skill, 1))['level']

    def bf(trait):
        return float(ratings.get(trait, 0.5))

    features = {
        'Computer Architecture': tech('Computer Architecture'),
        'Programming Skills':    tech('Programming Skills'),
        'Project Management':    tech('Project Management'),
        'Communication skills':  tech('Communication skills'),
        'Openness':              bf('Openness'),
        'Conscientousness':      bf('Conscientousness'),
        'Extraversion':          bf('Extraversion'),
        'Agreeableness':         bf('Agreeableness'),
        'Emotional_Range':       bf('Emotional_Range'),
        'Conversation':          bf('Conversation'),
        'Openness to Change':    bf('Openness to Change'),
        'Hedonism':              bf('Hedonism'),
        'Self-enhancement':      bf('Self-enhancement'),
        'Self-transcendence':    bf('Self-transcendence'),
    }
    return pd.DataFrame([features], columns=MODEL1_COLUMNS)


# ── Sanity check — verify score_row replication ───────────────────────────────
print('Sanity check: replicating score_row() from training...')
test_ratings = {
    'Deep Learning':                  9,   # → active
    'Natural Language Processing':    8,   # → active
    'Cyber Security':                 7,   # → active
    'Android App Development':        6,   # → active
    'Statistical Analysis':           1,   # → NOT active (None)
}
active = ratings_to_active_skills(test_ratings)
scores = score_row_from_active_skills(active)
non_zero = {k: v for k, v in scores.items() if v > 0}
print(f'  Active skills   : {sorted(active)}')
print(f'  Non-zero groups : {non_zero}')
print(f'  Sparse flag     : {scores["f_sparse_row"]} (expected 0)')

X2 = build_model2_features(test_ratings)
print(f'  Model 2 input shape: {X2.shape}  columns match feature_names: {list(X2.columns) == list(feature_names)}')
print('Feature builders defined ✓')

Sanity check: replicating score_row() from training...
  Active skills   : ['Android App Development', 'Cyber Security', 'Deep Learning', 'Natural Language Processing']
  Non-zero groups : {'f_nlp_deep_learning': 2, 'f_dev_mobile_android': 1, 'f_security_cyber': 1}
  Sparse flag     : 0 (expected 0)
  Model 2 input shape: (1, 27)  columns match feature_names: True
Feature builders defined ✓


## 6. Skill Gap Analysis (Model 2 artefacts)

In [11]:
def get_skill_gap(ratings: dict, predicted_career: str) -> dict:
    """
    Compare what the user has against what the predicted career requires,
    using career_profiles from Model 2 (built during training from >= 40% threshold).

    Returns
    -------
    missing_skills : list — required skills the user doesn't have (rating <= 1 / None)
    beginner_skills : list — required skills the user has at a beginner level (rating 2-5)
    intermediate_skills : list — required skills the user has at an intermediate level (rating 6-7)
    advanced_skills : list — required skills the user has at an advanced level (rating 8-9)
    ordinal_scores : dict — {feature_group: score} for the user's input
    """
    active_skills = ratings_to_active_skills(ratings)

    # Map canonical career name back to the key used in career_profiles
    PROFILE_KEY_MAP = {
        'Data Science & Analytics': 'Data Science',
    }
    profile_key = PROFILE_KEY_MAP.get(predicted_career, predicted_career)
    required = career_profiles.get(profile_key, set())
    if isinstance(required, list):
        required = set(required)

    missing_skills      = []
    beginner_skills     = []
    intermediate_skills = []
    advanced_skills     = []

    for s in sorted(required): # Sort for consistent output
        current_rating = ratings.get(s, 1) # Default to 1 if not rated
        proficiency = rating_to_proficiency(current_rating)['level']

        if proficiency == 0:  # Rating <= 1 (None)
            missing_skills.append(s)
        elif proficiency == 1: # Rating 2-5 (Beginner)
            beginner_skills.append(s)
        elif proficiency == 2: # Rating 6-7 (Intermediate)
            intermediate_skills.append(s)
        elif proficiency == 3: # Rating 8-9 (Advanced)
            advanced_skills.append(s)
        # Skills rated 10 (Mastered) will not be in any of these lists.

    # Ordinal group scores
    scores = score_row_from_active_skills(active_skills)
    ordinal_scores = {fn: scores.get(fn, 0) for fn in feature_names}

    return {
        'missing_skills': missing_skills,
        'beginner_skills': beginner_skills,
        'intermediate_skills': intermediate_skills,
        'advanced_skills': advanced_skills,
        'ordinal_scores': ordinal_scores,
    }

## 7. Ensemble Prediction Engine

In [15]:
# Model 1 (real data, calibrated LightGBM)    → higher weight
# Model 2 (generated data, reference XGBoost) → lower weight until real data collected
W1 = 0.35
W2 = 0.65
assert abs(W1 + W2 - 1.0) < 1e-9, 'Weights must sum to 1.0'


def ensemble_predict(
    ratings: dict,
    confidence_threshold: float = 0.35,
    w1: float = W1,
    w2: float = W2,
    verbose: bool = True,
) -> dict:
    """
    Predict career using a weighted average ensemble of both models.

    Parameters
    ----------
    ratings : dict
        Sub-skills from feature_schema  → int 1-10
        Big-Five traits                 → float 0.0-1.0
        Missing keys default to 1 (tech skills) or 0.5 (Big-Five).
    """
    # ── 1. Build feature DataFrames ──────────────────────────────────────────
    X1_df   = build_model1_features(ratings)
    X1_scaled = pd.DataFrame(
        scaler.transform(X1_df),
        columns=MODEL1_COLUMNS   # preserve column names → no LightGBM warning
    )
    X2_df = build_model2_features(ratings)

    # ── 2. Raw probability vectors ───────────────────────────────────────────
    proba1_raw = model1.predict_proba(X1_scaled)[0]
    proba2_raw = model2.predict_proba(X2_df)[0]

    # ── 3. Map to canonical career space ─────────────────────────────────────
    proba1_canon = map_probs_to_canonical(proba1_raw, le1, MODEL1_TO_CANONICAL, CANONICAL_CAREERS)
    proba2_canon = map_probs_to_canonical(proba2_raw, le2, MODEL2_TO_CANONICAL, CANONICAL_CAREERS)

    # ── 4. Weighted average ──────────────────────────────────────────────────
    ensemble_proba = w1 * proba1_canon + w2 * proba2_canon

    # ── 5. Rankings ──────────────────────────────────────────────────────────
    top3_idx    = np.argsort(ensemble_proba)[::-1][:3]
    top3        = [(CANONICAL_CAREERS[i], round(float(ensemble_proba[i]), 4)) for i in top3_idx]
    best_career = top3[0][0]
    confidence  = top3[0][1]
    model1_top  = CANONICAL_CAREERS[np.argmax(proba1_canon)]
    model2_top  = CANONICAL_CAREERS[np.argmax(proba2_canon)]

    # ── 6. Skill gap ─────────────────────────────────────────────────────────
    gap = get_skill_gap(ratings, best_career)

    # ── 7. DB record ─────────────────────────────────────────────────────────
    skill_summary = {}
    db_record     = {}
    for skill in ALL_SCHEMA_SKILLS:
        prof                 = rating_to_proficiency(ratings.get(skill, 1))
        skill_summary[skill] = prof['label']
        db_record[skill]     = prof['db_value']
    for trait in BIG_FIVE_TRAITS:
        val                  = float(ratings.get(trait, 0.5))
        skill_summary[trait] = f'{val:.2f}'
        db_record[trait]     = val

    # ── 8. Verbose breakdown ─────────────────────────────────────────────────
    if verbose:
        print('\n') # Print a newline explicitly first
        print('='*65)
        print('ENSEMBLE CAREER PREDICTION BREAKDOWN')
        print('='*65)
        print(f'\n  Weights  →  Model 1 (LightGBM): {w1:.0%}  |  Model 2 (XGBoost): {w2:.0%}')
        print(f'\n  Model 2 feature vector (XGBoost input):')
        for col in feature_names:
            val = X2_df[col].iloc[0]
            if val > 0:
                print(f'    {col:<35s}: {val}')
        print(f'\n  Model 1 top  ({w1:.0%}): {model1_top}')
        for i, c in enumerate(CANONICAL_CAREERS):
            print(f'    {c:<55s}: {proba1_canon[i]:.3f}')
        print(f'\n  Model 2 top  ({w2:.0%}): {model2_top}  [generated data — reference]')
        for i, c in enumerate(CANONICAL_CAREERS):
            print(f'    {c:<55s}: {proba2_canon[i]:.3f}')
        print(f'\n  Ensemble (weighted average):')
        for i, c in enumerate(CANONICAL_CAREERS):
            bar = '█' * int(ensemble_proba[i] * 30)
            print(f'    {c:<55s}: {ensemble_proba[i]:.3f}  {bar}')
        print('='*65)

    return {
        'predicted_career': best_career if confidence >= confidence_threshold
                            else 'Uncertain — please rate more skills',
        'confidence':       round(confidence, 4),
        'top_3':            top3,
        'model1_top':       model1_top,
        'model2_top':       model2_top,
        'models_agree':     model1_top == model2_top,
        'missing_skills':   gap['missing_skills'],
        'beginner_skills':  gap['beginner_skills'],
        'intermediate_skills': gap['intermediate_skills'],
        'advanced_skills':  gap['advanced_skills'],
        'ordinal_scores':   gap['ordinal_scores'],
        'input_quality':    'low' if gap['ordinal_scores'].get('f_sparse_row', 0) == 1 else 'ok',
        'skill_summary':    skill_summary,
        'db_record':        db_record,
    }

## 8. Interactive Survey

The survey is built **directly from `ALL_SCHEMA_SKILLS`** — no hard-coded list.
If you retrain Model 2 with new features, the survey updates automatically.

## 9. Run a Prediction

**Option A — interactive:**
```python
ratings = collect_ratings_interactive()
result  = ensemble_predict(ratings)
```

**Option B — dict directly** (demo below).

> ⚠️ Skill names must match **exactly** what `feature_schema` contains.
> Run Section 2 and copy names from `ALL_SCHEMA_SKILLS` to be safe.

In [13]:
def ask_user_proficiency_rating(skill_name: str) -> int:
    """
    Helper function to safely get a 1-10 proficiency rating for a chosen skill.
    Maps directly to the notebook's underlying proficiency breakdown.
    """
    print(f"  ⭐ Rate your proficiency for '{skill_name}':")
    print("     [1-2] None | [3-5] Beginner | [6-7] Intermediate | [8-9] Advanced | [10] Mastered")
    while True:
        try:
            rating = int(input("     👉 Enter rating (1-10): "))
            if 1 <= rating <= 10:
                return rating
            print("     ❌ Rating must be an integer between 1 and 10.")
        except ValueError:
            print("     ❌ Please enter a valid integer.")


def run_precise_skills_survey():
    print("=" * 60)
    print("🎓 MULTIMODAL ENSEMBLE CAREER PREDICTION SURVEY 🎓")
    print("=" * 60)
    print("Select your skills, rate your proficiency, and input your personality traits.\n")

    # Initialize ALL valid schema skills to 1 (None) so unselected skills default safely
    ratings_dict = {skill: 1 for skill in ALL_SCHEMA_SKILLS}

    # ── STEP 1: Core Track Depths ──────────────────────────────────────────
    print("--- 🗺️ CORE CAREER TRACK DEPTHS ---")
    for group_key, meta in skill_gap_map.items():
        print(f"\n🔹 {meta['ordinal_label']}:")
        for level_score, level_skills, level_label in meta['levels']:
            clean_skills = [s for s in level_skills if s != "Web Development"]
            print(f"  [{level_score}] {level_label}")
            if clean_skills:
                print(f"      (Gives skills: {', '.join(clean_skills)})")

        max_level = max(level[0] for level in meta['levels'])
        while True:
            try:
                choice = int(input(f"👉 Enter choice (0-{max_level}): "))
                if 0 <= choice <= max_level:
                    for level_score, level_skills, _ in meta['levels']:
                        if level_score == choice:
                            clean_skills = [s for s in level_skills if s != "Web Development"]
                            # Dynamically prompt for ratings on skills granted by this track selection
                            if clean_skills:
                                print(f"\n📋 Please rate the skills granted by this track:")
                                for s in clean_skills:
                                    ratings_dict[s] = ask_user_proficiency_rating(s)
                    break
                print(f"❌ Selection out of bounds.")
            except ValueError:
                print("❌ Enter a valid integer.")

    # ── STEP 2: Additional Tech & Soft Skill Groups ───────────────────────
    # Track which skills have already been evaluated so we don't display them again
    covered_skills = set(
        skill for meta in skill_gap_map.values()
        for level in meta['levels'] for skill in level[1]
    )

    print("\n" + "─" * 40)
    print("--- 🛠️ ADDITIONAL TECH & SOFT SKILL GROUPS ---")
    print("Enter the numbers of ALL sub-skills you possess, separated by commas (e.g., 1,3,4).")
    print("If you have none of them, simply enter 0.")

    for feature_name, sub_skills in feature_schema.items():
        if feature_name == 'f_sparse_row' or not sub_skills:
            continue

        display_skills = [s for s in sub_skills if s not in covered_skills]
        if not display_skills:
            continue

        clean_label = feature_name.replace("f_", "").replace("_", " ").title()
        print(f"\n🔹 {clean_label}:")
        for i, s in enumerate(display_skills, 1):
            print(f"  {i}. {s}")

        while True:
            user_input = input(f"👉 Select your skills (or 0 for none): ").strip()
            if user_input == '0':
                break
            try:
                choices = [int(x.strip()) for x in user_input.split(',') if x.strip()]
                if all(1 <= c <= len(display_skills) for c in choices):
                    print(f"\n📋 Please rate your selected skills:")
                    for c in choices:
                        s = display_skills[c - 1]
                        ratings_dict[s] = ask_user_proficiency_rating(s)
                    break
                print(f"❌ Invalid choice. Please use numbers between 1 and {len(display_skills)}.")
            except ValueError:
                print("❌ Please enter numbers separated by commas (e.g., 1,2).")

    # ── STEP 3: Handle Big-Five Personality Traits (Crucial for Model 1) ───
    print("\n" + "─" * 40)
    print("--- 🧠 PERSONALITY & BIG-FIVE TRAITS (FOR MODEL 1) ---")
    print("Enter a float value between 0.0 and 1.0 for each trait (e.g., 0.65).")

    for trait in BIG_FIVE_TRAITS:
        clean_trait_label = trait.replace("_", " ").title()
        while True:
            try:
                val = float(input(f"👉 Enter value for {clean_trait_label} (0.0 - 1.0): "))
                if 0.0 <= val <= 1.0:
                    ratings_dict[trait] = val
                    break
                print("❌ Value must be strictly between 0.0 and 1.0.")
            except ValueError:
                print("❌ Please enter a valid decimal number.")

    # ── STEP 4: Call Ensemble Prediction Engine ─────────────────────────────
    print("\n" + "=" * 60)
    print("🔄 Processing your profile and running prediction metrics...")
    print("=" * 60)

    # Pass the fully built ratings dictionary directly to the ensemble engine
    # Set verbose=True to let the ensemble function display its beautiful built-in logs
    prediction_result = ensemble_predict(ratings_dict, verbose=True)

    # Post-process 'missing_skills' to clean global noise out of the display output
    NOISE_SKILLS = {'Adaptability', 'Communication', 'Critical Thinking', 'Graphic Design', 'Web Development'}

    # Safely extract results regardless of exact key naming styles
    predicted_career = prediction_result.get('predicted_career', 'Unknown')
    confidence = prediction_result.get('confidence', 0.0)
    top_3 = prediction_result.get('top_3', [])
    missing_skills = [s for s in prediction_result.get('missing_skills', []) if s not in NOISE_SKILLS]
    beginner_skills = [s for s in prediction_result.get('beginner_skills', []) if s not in NOISE_SKILLS]
    intermediate_skills = [s for s in prediction_result.get('intermediate_skills', []) if s not in NOISE_SKILLS]
    advanced_skills = [s for s in prediction_result.get('advanced_skills', []) if s not in NOISE_SKILLS]
    ordinal_scores = prediction_result.get('ordinal_scores', {})

    # ── STEP 5: Final Clean Summary Output ──────────────────────────────────
    print("\n" + "🎯" * 30)
    print(f"🎯 FINAL CONSOLIDATED ENSEMBLE PREDICTION")
    print("" + "🎯" * 30)
    print(f"  🏆 Career Target   : {predicted_career}")
    print(f"  📈 Confidence Score : {confidence:.1%}")
    print(f"  📊 Top 3 Candidates : {top_3}")
    print(f"  ❌ Missing Skills   : {missing_skills}")
    print(f"  👶 Beginner Skills : {beginner_skills}")
    print(f"  📖 Intermediate Skills : {intermediate_skills}")
    print(f"  🎓 Advanced Skills : {advanced_skills}")
    print(f"  📉 Feature Densities: {list(ordinal_scores.items())[:4]}... (see breakdown above)")
    print("=" * 60 + "\n")

    return prediction_result

## 11. Adjusting Weights When Real Data Arrives

```python
# Equal footing after retraining Model 2 on real data:
result = ensemble_predict(ratings, w1=0.50, w2=0.50)
```

Or tune automatically on a validation set:
```python
from sklearn.metrics import accuracy_score
best_w1, best_acc = 0.5, 0
for w1 in np.arange(0.1, 1.0, 0.05):
    preds = [ensemble_predict(r, w1=w1, w2=round(1-w1,2), verbose=False)['predicted_career']
             for r in val_ratings]
    acc = accuracy_score(val_labels, preds)
    if acc > best_acc:
        best_acc, best_w1 = acc, w1
print(f'Best w1={best_w1:.2f}  Accuracy={best_acc:.3f}')
```

In [16]:
run_precise_skills_survey()

🎓 MULTIMODAL ENSEMBLE CAREER PREDICTION SURVEY 🎓
Select your skills, rate your proficiency, and input your personality traits.

--- 🗺️ CORE CAREER TRACK DEPTHS ---

🔹 Mobile Development Coverage:
  [0] No mobile skills
  [1] Android only
      (Gives skills: Android App Development)
  [2] iOS only
      (Gives skills: iOS App Development)
  [3] Cross-platform / general
      (Gives skills: Mobile App Development)


KeyboardInterrupt: Interrupted by user